# GPT

> - The GPT architecture on which this project is based is not publicly open-sourced; this implementation is inferred and reproduced from the official technical reports.
>   
> - The final design is closer to a GPT-3.5–style architecture and incorporates several engineering practices from early community implementations based on guesses about the original GPT architecture.
>
> - This project is intended as an exercise in understanding the architecture and implementation details. It is provided only as Jupyter Lab notebooks (no standalone Python scripts), but the correctness of the architectural data flow has been validated.

## Architecture

Our implementation is based on the following architecture:

<center><img src="https://skojiangdoc.oss-cn-beijing.aliyuncs.com/2024LLM/17.png" alt="描述文字" width="300">

## Implementation

In [4]:
import math
import torch
from torch import nn
import torch.nn.functional as F
from typing import Any, Optional, Tuple
from transformers import PretrainedConfig
from transformers import PreTrainedModel
import torch.utils.checkpoint as checkpoint
from transformers.modeling_outputs import CausalLMOutputWithPast

In [16]:
class LMConfig(PretrainedConfig):
    model_type = "MateConv_GPT"
    
    def __init__(
        self,
        vocab_size=6400, 
        max_seq_len=1024, 
        dim=768,
        hidden_dim=3072,
        n_heads=12,
        n_layers=12,
        dropout=0.1,
        norm_eps=1e-5,
        use_checkpoint=False,
        **kwargs
    ):

        super().__init__(**kwargs)
        
        self.vocab_size = vocab_size  # 词汇表大小
        self.max_seq_len = max_seq_len  # 最大序列长度
        self.dim = dim  # 嵌入维度
        self.hidden_dim = hidden_dim  # 前馈网络隐藏层维度
        self.n_heads = n_heads  # 注意力头数量
        self.n_layers = n_layers  # TransformerBlock 的层数（串联解码器的个数）
        self.dropout = dropout  # Dropout 概率
        self.norm_eps = norm_eps  # LayerNorm 的 epsilon
        self.use_checkpoint = use_checkpoint  # 是否使用梯度检查点

        # 计算每个注意力头的维度，确保可以被整除
        if dim % n_heads != 0:
            raise ValueError("`dim` must be divisible by `n_heads`.")
        self.head_dim = dim // n_heads

        W = math.ceil(math.sqrt(max_seq_len))
        H = math.ceil(max_seq_len / W)
        
        self.axial_h = int(H)
        self.axial_w = int(W)

In [6]:
# 轴向位置编码
def axial_positional_emb(embedding_dim, axial_dim_1, axial_dim_2):
    axial_wpe_1 = nn.Parameter(torch.randn(axial_dim_1, embedding_dim) * 0.01)
    axial_wpe_2 = nn.Parameter(torch.randn(axial_dim_2, embedding_dim) * 0.01)

    # 广播并组合轴向位置编码
    axial_wpe_1 = axial_wpe_1.unsqueeze(1).expand(-1, axial_dim_2, embedding_dim)
    axial_wpe_2 = axial_wpe_2.unsqueeze(0).expand(axial_dim_1, -1, embedding_dim)
    wpe = (axial_wpe_1 + axial_wpe_2) / 2

    return wpe.view(axial_dim_1 * axial_dim_2, embedding_dim)

In [7]:
class MLP_GLU(nn.Module):
    def __init__(self, dim: int, hidden_dim: int, dropout: float):
        super().__init__()
        self.w1 = nn.Linear(dim, 2 * hidden_dim, bias=False)
        self.w2 = nn.Linear(hidden_dim, dim, bias=False)
        self.dropout = nn.Dropout(dropout)

    def forward(self, x):
        h = self.w1(x)
        h, gate = h.chunk(2, dim=-1)
        h = h * torch.sigmoid(gate)
        h2 = self.w2(h)
        return self.dropout(h2)

In [19]:
# Transformer Block（基于 MacaronAttention）
class TransformerBlock(nn.Module):
    def __init__(self, layer_id: int, config: LMConfig):
        super().__init__()
        self.layer_id = layer_id

        # qkv
        self.linear1 = nn.Linear(config.dim, config.dim)
        self.linear2 = nn.Linear(config.dim, config.dim)
        self.linear3 = nn.Linear(config.dim, config.dim)
        
        # 自注意力
        self.attention = nn.MultiheadAttention(embed_dim=config.dim, num_heads=config.n_heads, dropout=config.dropout, batch_first=True)

        # GLU 的双前馈层
        self.ffn1 = MLP_GLU(config.dim, config.hidden_dim, config.dropout)
        self.ffn2 = MLP_GLU(config.dim, config.hidden_dim, config.dropout)

        # ReZero 参数
        self.alpha1 = nn.Parameter(torch.tensor(0.0))
        self.alpha2 = nn.Parameter(torch.tensor(0.0))
        self.alpha3 = nn.Parameter(torch.tensor(0.0))

        # LayerNorm 和 Dropout
        self.norm1 = nn.LayerNorm(config.dim, eps=config.norm_eps)
        self.norm2 = nn.LayerNorm(config.dim, eps=config.norm_eps)
        self.norm3 = nn.LayerNorm(config.dim, eps=config.norm_eps)
        self.dropout = nn.Dropout(config.dropout)

    def forward(self, x, pos_cis):
        # 前馈网络 1
        residual = x
        x = self.norm1(x)
        x = self.ffn1(x)
        x = residual + self.alpha1 * self.dropout(x)

        # 多头注意力
        residual = x
        x = self.norm2(x)
        q = self.linear1(x)
        k = self.linear2(x)
        v = self.linear3(x)
        x, _ = self.attention(q, k, v)
        x = residual + self.alpha2 * self.dropout(x)

        # 前馈网络 2
        residual = x
        x = self.norm3(x)
        x = self.ffn2(x)
        x = residual + self.alpha3 * self.dropout(x)

        return x

In [22]:
# GPT 模型
class GPT(PreTrainedModel):
    config_class = LMConfig  # 定义模型使用的配置类
    last_loss: Optional[torch.Tensor]  # 用于记录最后计算的损失值
    
    def __init__(self, params: LMConfig = None):
        """
        GPT 是一个基于 Transformer 架构的语言模型，继承自 PreTrainedModel。

        参数：
        - params: 配置对象 LMConfig，包含模型的超参数配置。
        """
        super().__init__(params)
        if not params:
            params = LMConfig()  # 如果没有提供配置，则使用默认配置
        self.params = params  # 保存模型参数配置
        self.vocab_size = params.vocab_size  # 词汇表大小
        self.n_layers = params.n_layers  # Transformer 的层数

        # 词嵌入和轴向位置编码
        self.embedding = nn.Embedding(config.vocab_size, config.dim)
        self.dropout = nn.Dropout(config.dropout)
        self.pos_emb = axial_positional_emb(config.dim, config.axial_h, config.axial_w)

        # Transformer 层
        self.blocks = nn.ModuleList([TransformerBlock(i, config) for i in range(config.n_layers)]) # nn.sequential只能按顺序串联
        self.norm = nn.LayerNorm(config.dim, eps=config.norm_eps)

        # 输出层：共享嵌入权重
        self.output = nn.Linear(config.dim, config.vocab_size, bias=False)
        self.output.weight = self.embedding.weight  # 共享词嵌入层和输出层的权重

        #初始化模型权重
        self.apply(self._init_weights)  # 初始化模型权重
        self.OUT = CausalLMOutputWithPast()  # 初始化OUT类
        #OUT类用于封装模型的输出信息，它不参与模型的计算，而是帮助管理和返回输出结果

    def _init_weights(self, module): # 只在内部调用的函数
        """
        初始化模块权重。
        - 对线性层使用正态分布初始化权重，均值为 0，标准差为 0.02。
        - 对词嵌入层也使用正态分布初始化权重，均值为 0，标准差为 0.02。
        """
        if isinstance(module, nn.Linear):
            torch.nn.init.normal_(module.weight, mean=0.0, std=0.02)
            if module.bias is not None:
                torch.nn.init.zeros_(module.bias)
        elif isinstance(module, nn.Embedding):
            torch.nn.init.normal_(module.weight, mean=0.0, std=0.02)
    
    def forward(self, tokens: Optional[torch.Tensor] = None
                , targets: Optional[torch.Tensor] = None
                , **keyargs):
        """
        GPT 的前向传播函数。

        参数：
        - tokens: 输入的 token 张量，表示输入的词序列。
        - targets: 目标张量，用于计算交叉熵损失。
        - keyargs: 其他可选参数，如 'input_ids' 和 'attention_mask'。

        返回：
        - 输出的 logits 和 loss（如果有目标）。
        """
        current_idx = 0  # 当前索引初始化为 0
        if 'input_ids' in keyargs:
            tokens = keyargs['input_ids']  # 从关键字参数中提取 'input_ids'
        if 'attention_mask' in keyargs:
            attention_mask = keyargs['attention_mask'] # 从关键字参数中提取 'attention_mask'
        if 'targets' in keyargs:
            targets = keyargs['targets']  
        if 'current_idx' in keyargs:
            current_idx = int(keyargs['current_idx'])  # 更新当前索引

        # 获取输入 tokens 的序列长度
        seq_len = tokens.size(1)

        # 嵌入层 + 位置编码 + dropout
        h = self.embedding(tokens) + self.pos_emb[:seq_len, :].to(tokens.device)
        h = self.dropout(h)

        # 梯度检查点技术
        for idx, block in enumerate(self.blocks): # 取出编号和本身
            if self.params.use_checkpoint and idx % 4 != 0:  # 使用梯度检查点，每 4 个保存一次激活（不能除尽的chekcpoint，不保存）
                h = checkpoint.checkpoint(block, h, self.pos_emb.to(tokens.device))
            else:  # 完整保存激活值
                h = block(h, self.pos_emb.to(tokens.device))
        
        # 最后的layer norm
        h = self.norm(h)
        
        if targets is not None:
            logits = self.output(h)  # 通过线性输出层生成 logits
            # 计算交叉熵损失，忽略 index 为 0 的位置，reduction 为 'none'，即不自动求平均
            self.last_loss = F.cross_entropy(logits.view(-1, logits.size(-1)), targets.view(-1),
                                             ignore_index=0, reduction='none')
        else:
            logits = self.output(h[:, [-1], :])  # 如果没有目标，只返回最后一个时间步的 logits
            self.last_loss = None  # 没有损失
        
        self.OUT.__setitem__('logits', logits)  # 设置输出的 logits
        self.OUT.__setitem__('last_loss', self.last_loss)  # 设置最后的 loss
        return self.OUT  # 返回输出对象

    @torch.inference_mode()
    def generate(self, idx, eos, max_new_tokens, temperature=0.7, top_k=8, stream=True, rp=1., kv_cache=True):
        """
        推理模式下的文本生成函数。

        参数：
        - idx: 输入的 tokens。
        - eos: 结束标志符号，当生成到 eos 时停止生成。
        - max_new_tokens: 最大生成的新 token 数量。
        - temperature: 控制生成的随机性，温度越高，生成越多样化。
        - top_k: 限制 top-k 采样，控制只选择概率最高的 k 个 token。
        - stream: 是否进行流式输出。
        - rp: 重复惩罚系数，控制重复 token 的惩罚。
        - kv_cache: 是否使用键值缓存来加速推理。

        返回：
        - 生成的 tokens（可能是流式返回）。
        """
        index = idx.shape[1]  # 获取输入 token 序列的长度
        init_inference = True  # 初始化推理标志
        while idx.shape[1] < max_new_tokens - 1:  # 当生成的 tokens 长度小于最大 tokens 数时继续生成
            if init_inference or not kv_cache:
                inference_res, init_inference = self(idx, kv_cache=kv_cache), False  # 第一次推理，或不使用缓存
            else:
                inference_res = self(idx[:, -1:], kv_cache=kv_cache, current_idx=idx.shape[1] - 1)  # 仅使用最后一个 token 推理

            logits = inference_res.logits  # 获取推理结果的 logits
            logits = logits[:, -1, :]  # 只选择最后一个 token 的 logits

            # 对生成的 token 进行重复惩罚
            for token in set(idx.tolist()[0]):
                logits[:, token] /= rp  # 对每个重复的 token 施加惩罚

            if temperature == 0.0:  # 如果温度为 0，使用贪心算法选择下一个 token
                _, idx_next = torch.topk(logits, k=1, dim=-1)
            else:
                logits = logits / temperature  # 根据温度调整 logits
                if top_k is not None:
                    v, _ = torch.topk(logits, min(top_k, logits.size(-1)))  # 使用 top-k 采样
                    logits[logits < v[:, [-1]]] = -float('Inf')  # 排除 top-k 之外的 logits

                probs = F.softmax(logits, dim=-1)  # 计算概率分布
                idx_next = torch.multinomial(probs, num_samples=1, generator=None)  # 根据概率进行采样

            if idx_next == eos:  # 如果生成了 eos token，停止生成
                break

            idx = torch.cat((idx, idx_next), dim=1)  # 将生成的 token 拼接到输入序列中
            if stream:  # 如果启用了流式输出
                yield idx[:, index:]  # 输出当前生成的 tokens

        if not stream:  # 如果未启用流式输出
            yield idx[:, index:]  # 返回生成的完整序列

## Validation

We simulate an input with batch size 1 and sequence length 10 tokens to verify that the model architecture is wired correctly.

In [23]:
config = LMConfig()
model = GPT(config)

# 测试输入
tokens = torch.randint(0, config.vocab_size, (1, 10))  
targets = torch.randint(0, config.vocab_size, (1, 10))

# 测试前向传播
output = model(tokens=tokens, targets=targets)
print("Logits:", output['logits'])
print("Loss:", output['last_loss'])

# 测试生成
print("\nGenerated Sequence:")
eos_token = 6004968  # 假设 50256 是结束符
for generated in model.generate(tokens, eos=eos_token, top_k = 30, max_new_tokens=20, temperature=0.7):
    print(generated)

Logits: tensor([[[-0.6805, -0.2615,  0.6100,  ...,  0.6947,  0.5500, -1.2819],
         [ 0.2497, -0.5095, -0.6377,  ..., -0.2183, -0.6717, -0.1640],
         [ 0.1888,  0.5717,  0.4119,  ..., -0.2443, -0.0426,  0.3392],
         ...,
         [-0.1814, -0.5048, -0.5225,  ...,  0.1005,  0.2810, -0.9258],
         [ 0.2557,  0.4936, -0.2230,  ..., -0.0250, -0.4641, -0.9499],
         [-0.2724,  0.6432, -0.7532,  ...,  0.1704,  0.0511,  0.5121]]],
       grad_fn=<UnsafeViewBackward0>)
Loss: tensor([13.3889, 11.5518, 13.3064, 13.0774, 13.1622, 13.7653, 13.3287, 13.3480,
        15.0269, 14.4225], grad_fn=<NllLossBackward0>)

Generated Sequence:
tensor([[1718]])
tensor([[1718, 1718]])
tensor([[1718, 1718, 1718]])
tensor([[1718, 1718, 1718, 1718]])
tensor([[1718, 1718, 1718, 1718, 1718]])
tensor([[1718, 1718, 1718, 1718, 1718, 1718]])
tensor([[1718, 1718, 1718, 1718, 1718, 1718, 1718]])
tensor([[1718, 1718, 1718, 1718, 1718, 1718, 1718, 1718]])
tensor([[1718, 1718, 1718, 1718, 1718, 1718, 1

- The logits shape, loss length, and the step-by-step growth of the generated sequence are all consistent with the expected data flow.

- The model repeatedly generating the token `1718` is also expected behavior at this stage, since the weights have not been trained yet.